# Tool Size Distribution

**Purpose**: compute the absolute tool-size distribution across labelled masks (foreground area as % of image),
and report percentiles used as physical thresholds elsewhere in the dissertation.

**Input**: Pascal VOC segmentation masks (`SegmentationClass/`) plus `labelmap.txt`.
Uses local paths when present; otherwise pulls from GCS (`gs://laparoscopic-seg-REPLACE_K_NUMBER/trainingFrames/`).

**Output**: percentile table + histogram of absolute tool sizes, then frame counts under alternative banding schemes.


## 1. Install dependencies
Run this cell to install the packages required for this notebook.

In [ ]:
%pip install -q matplotlib tqdm pandas opencv-python-headless

import cv2, matplotlib, pandas as pd, numpy as _np
print(f"cv2 {cv2.__version__} | matplotlib {matplotlib.__version__} "
      f"| pandas {pd.__version__} | numpy {_np.__version__}")

## 2. Setup and configuration
Resolves the mask directory and labelmap path (local first, GCS fallback), then sets the
foreground label. Override with environment variables if the files move.

In [ ]:
import os
import glob
import shutil
import subprocess
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Paths ────────
# Override with environment variables if the files move.
GCS_BUCKET = os.getenv("LAP_GCS_BUCKET", "laparoscopic-seg-REPLACE_K_NUMBER")
GCS_PREFIX = f"gs://{GCS_BUCKET}/trainingFrames"
CACHE_DIR = Path(os.getenv("LAP_CACHE_DIR", Path.home() / ".cache" / "laparoscopic_seg"))

DEFAULT_DATA_ROOT = Path.home() / "Desktop" / "Individual Project" / "combined"
MASKS_DIR = Path(os.getenv("LAP_MASKS_DIR", DEFAULT_DATA_ROOT / "SegmentationClass"))
FOREGROUND_LABEL = os.getenv("LAP_FOREGROUND_LABEL", "instrument")


def _count_pngs(root: Path) -> int:
    return len(glob.glob(str(root / "**" / "*.png"), recursive=True))


def _gsutil_available() -> bool:
    return shutil.which("gsutil") is not None


def _gsutil_cp(src: str, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gsutil", "cp", src, str(dst)], check=True)


def _gsutil_rsync(src: str, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gsutil", "-m", "rsync", "-r", src, str(dst)], check=True)


def resolve_labelmap() -> Path:
    """Local candidates first; download from GCS into the cache if none exist."""
    if os.getenv("LAP_LABELMAP"):
        candidate = Path(os.getenv("LAP_LABELMAP"))
        if candidate.is_file():
            return candidate
        raise FileNotFoundError(f"LAP_LABELMAP set but not found: {candidate}")

    for candidate in [
        MASKS_DIR.parent / "labelmap.txt",
        Path.cwd() / "labelmap.txt",
        Path.home() / "VSCodeProjects" / "dissertation" / "archive" / "examples" / "labelmap.txt",
        DEFAULT_DATA_ROOT / "labelmap.txt",
    ]:
        if candidate.is_file():
            return candidate

    if not _gsutil_available():
        raise FileNotFoundError(
            "labelmap.txt not found locally and gsutil is unavailable.\n"
            "Set LAP_LABELMAP, or install the Google Cloud SDK."
        )
    cached = CACHE_DIR / "labelmap.txt"
    print(f"Downloading labelmap from {GCS_PREFIX}/labelmap.txt ...")
    _gsutil_cp(f"{GCS_PREFIX}/labelmap.txt", cached)
    return cached


def resolve_masks_dir() -> Path:
    """Use local masks if present; otherwise rsync SegmentationClass from GCS."""
    if MASKS_DIR.is_dir() and _count_pngs(MASKS_DIR) > 0:
        return MASKS_DIR

    if os.getenv("LAP_MASKS_DIR") and not MASKS_DIR.is_dir():
        raise FileNotFoundError(f"LAP_MASKS_DIR set but not found: {MASKS_DIR}")

    if not _gsutil_available():
        raise FileNotFoundError(
            f"Masks dir not found (or empty): {MASKS_DIR}\n"
            "Set LAP_MASKS_DIR, or install gsutil to pull from GCS."
        )
    cached = CACHE_DIR / "SegmentationClass"
    if _count_pngs(cached) == 0:
        print(f"Downloading masks from {GCS_PREFIX}/SegmentationClass ...")
        _gsutil_rsync(f"{GCS_PREFIX}/SegmentationClass", cached)
    else:
        print(f"Using cached masks ({_count_pngs(cached)} pngs): {cached}")
    return cached


LABELMAP_FILE = resolve_labelmap()
MASKS_DIR = resolve_masks_dir()

n_masks = _count_pngs(MASKS_DIR)
print("Masks dir:", MASKS_DIR)
print("Labelmap:", LABELMAP_FILE)
print(f"Foreground label: {FOREGROUND_LABEL}")
print(f"Mask PNGs found: {n_masks}")
assert n_masks > 0, f"No .png masks under {MASKS_DIR}"
assert LABELMAP_FILE.is_file(), f"Labelmap not found: {LABELMAP_FILE}"

## 3. Parse labelmap
Reads `labelmap.txt` to map class names to RGB colours. The colour corresponding to
`FOREGROUND_LABEL` is used for the binary mask conversion.

In [ ]:
def parse_labelmap(path: Path) -> dict:
    """Return {label_name: (r, g, b)} from a Pascal VOC segmentation format (`name:r,g,b::`)."""
    label_colours = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            name, rgb_part, *_ = line.split(":")
            r, g, b = [int(x) for x in rgb_part.split(",")]
            label_colours[name] = (r, g, b)
    return label_colours

label_colours = parse_labelmap(LABELMAP_FILE)
print("=" * 60)
for name, (r, g, b) in label_colours.items():
    print(f"\033[48;2;{r};{g};{b}m    \033[0m {name:12s} : ({r}, {g}, {b})")

assert FOREGROUND_LABEL in label_colours, (
    f"'{FOREGROUND_LABEL}' not in labelmap. Available: {list(label_colours)}"
)

FG_COLOUR = np.array(label_colours[FOREGROUND_LABEL], dtype=np.uint8)
print(f"foreground     : '{FOREGROUND_LABEL}' RGB{tuple(FG_COLOUR)}")
print("=" * 60)

## 4. Tool size distribution (absolute % of image)
Scan every mask, measure foreground area as a fraction of the frame, then print a percentile table and plot the histogram.


In [ ]:
mask_files = glob.glob(str(MASKS_DIR / "**" / "*.png"), recursive=True)

sizes = []

print(f"Calculating sizes for {len(mask_files)} masks...")
for mask_file in tqdm(mask_files):
    mask_rgb = cv2.cvtColor(cv2.imread(mask_file), cv2.COLOR_BGR2RGB)
    total_pixels = mask_rgb.shape[0] * mask_rgb.shape[1]
    fg_pixels = np.sum(np.all(mask_rgb == FG_COLOUR, axis=-1))
    if fg_pixels > 0:
        fraction = fg_pixels / total_pixels
        sizes.append(fraction)

sizes_series = pd.Series(sizes) * 100


def dist_table(series, name, decimals=2):
    s = series.describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90])
    df = pd.DataFrame([{
        "n": int(s["count"]),
        "mean": s["mean"],
        "std": s["std"],
        "min": s["min"],
        "p5": s["5%"],
        "p10": s["10%"],
        "p25": s["25%"],
        "p50": s["50%"],
        "p75": s["75%"],
        "p90": s["90%"],
        "max": s["max"],
    }])
    print(f"\n--- {name} ---")
    fmt = lambda v, d=decimals: f"{v:.{d}f}"
    print(df.to_string(index=False, formatters={
        "n": "{:.0f}".format,
        **{c: fmt for c in df.columns if c != "n"},
    }))
    return s


stats = dist_table(sizes_series, "Tool Size Distribution (Absolute % of Image)")
print(f"plot line: p10 = {stats['10%']:.2f}%")

plt.figure(figsize=(10, 5))
plt.hist(sizes_series, bins=50, color='red', edgecolor='white')
plt.title("Distribution of Absolute Tool Sizes")
plt.xlabel("Percentage of Image Area (%)")
plt.ylabel("Frequency (Frames)")
plt.axvline(sizes_series.quantile(0.10), color='#333333', linestyle='dashed', linewidth=2, label='10th Percentile')
plt.legend()
plt.show()


## 5. Rejected alternatives
Two four-band schemes compared with the 4.5 / 8 / 19 cuts used for `size_cls`.

1. **Equal-count** - sort frames by tool area and split into four groups of the same size.
2. **Equal-width** - split `[min, max]` into four equal area-percentage quarters.


In [ ]:
n = len(sizes_series)
mx = float(sizes_series.max())


def band_table(edges, name):
    counts, e = np.histogram(sizes_series, bins=edges)
    rows = []
    for i, c in enumerate(counts):
        closed = "]" if i == len(counts) - 1 else ")"
        rows.append({
            "band": f"[{e[i]:.1f}, {e[i + 1]:.1f}{closed}",
            "frames": int(c),
            "pct": 100.0 * c / n,
        })
    df = pd.DataFrame(rows)
    print(f"\n--- {name} ---")
    print(df.to_string(index=False, formatters={"pct": "{:.1f}%".format}))
    i_hi, i_lo = int(df["frames"].idxmax()), int(df["frames"].idxmin())
    print(f"densest: {df.loc[i_hi, 'pct']:.1f}%   sparsest: {int(df.loc[i_lo, 'frames'])} frames")
    return df


# 1. Equal-count: four rank-order groups.
qcat = pd.qcut(sizes_series, 4)
q_rows = []
for iv, c in qcat.value_counts().sort_index().items():
    q_rows.append({
        "band": f"[{iv.left:.2f}, {iv.right:.2f}]",
        "frames": int(c),
        "pct": 100.0 * c / n,
    })
q_df = pd.DataFrame(q_rows)
print("\n--- Equal-count (4 rank-order groups) ---")
print(q_df.to_string(index=False, formatters={"pct": "{:.1f}%".format}))
print(f"lowest band reaches {qcat.cat.categories[0].right:.1f}% (p25)")

# 2. Equal-width: [min, max] split into four equal area-percentage quarters.
eq_edges = np.linspace(float(sizes_series.min()), mx, 5)
_ = band_table(eq_edges, "Equal-width (min-max split into quarters)")

_ = band_table(np.array([0.0, 4.5, 8.0, 19.0, mx]), "Chosen cuts (4.5 / 8 / 19)")